[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/10_decision_trees/02_classifier_from_scratch.ipynb)

# Bài 02 — Cài đặt Decision Tree (CART) from scratch

**🎯 Mục tiêu**
- Tìm split tốt nhất cho một feature: từ cách "thử mọi ngưỡng" $O(n^2)$ → sắp xếp + tổng tích luỹ $O(n\log n)$.
- Xây cây đệ quy với các điều kiện dừng (`max_depth`, `min_samples_split`, `min_samples_leaf`).
- Dự đoán, in cây dạng luật if/else, tính **feature importance** (MDI).
- Đối chiếu với `sklearn.tree.DecisionTreeClassifier`.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/10_decision_trees

In [ ]:
import time
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from ml_course.datasets import make_classification_2d
from ml_course.visualization import plot_decision_boundary, set_style

set_style()

## 1. Split tốt nhất cho một feature

**Cách 1 — thử mọi ngưỡng:** ngưỡng ứng viên là trung điểm giữa hai giá trị liên tiếp (đã sắp xếp); với mỗi ngưỡng, đếm lại nhãn hai bên → $O(n)$ mỗi ngưỡng, $O(n^2)$ tổng.

In [ ]:
def gini_of_counts(counts):
    n = counts.sum()
    return 1 - np.sum((counts / n) ** 2) if n > 0 else 0.0


def best_split_naive(x, y, n_classes):
    values = np.unique(x)
    thresholds = (values[:-1] + values[1:]) / 2
    parent = gini_of_counts(np.bincount(y, minlength=n_classes))
    best_t, best_gain = None, 0.0
    for t in thresholds:
        left, right = y[x <= t], y[x > t]
        child = (len(left) * gini_of_counts(np.bincount(left, minlength=n_classes))
                 + len(right) * gini_of_counts(np.bincount(right, minlength=n_classes))) / len(y)
        if parent - child > best_gain:
            best_t, best_gain = t, parent - child
    return best_t, best_gain

**Cách 2 — sắp xếp + tổng tích luỹ:** sắp xếp $x$ một lần; khi ngưỡng trượt qua từng điểm, số đếm lớp bên trái tăng đúng 1 → dùng `cumsum` của one-hot nhãn để có số đếm hai bên cho **mọi** ngưỡng cùng lúc.

In [ ]:
def best_split_fast(x, y, n_classes):
    order = np.argsort(x, kind="mergesort")
    xs, ys = x[order], y[order]
    n = len(y)
    onehot = np.eye(n_classes)[ys]
    left_counts = np.cumsum(onehot, axis=0)[:-1]          # vị trí cắt sau phần tử i
    right_counts = onehot.sum(0) - left_counts
    n_left = np.arange(1, n)[:, None]
    n_right = n - n_left
    gini_left = 1 - ((left_counts / n_left) ** 2).sum(1)
    gini_right = 1 - ((right_counts / n_right) ** 2).sum(1)
    child = (n_left[:, 0] * gini_left + n_right[:, 0] * gini_right) / n
    valid = xs[1:] != xs[:-1]                              # chỉ cắt giữa hai giá trị khác nhau
    child[~valid] = np.inf
    i = int(np.argmin(child))
    parent = 1 - ((onehot.sum(0) / n) ** 2).sum()
    return (xs[i] + xs[i + 1]) / 2, parent - child[i]


rng = np.random.default_rng(0)
x = rng.normal(size=3000)
y = (x + rng.normal(0, 0.8, 3000) > 0.3).astype(int)
for f in [best_split_naive, best_split_fast]:
    start = time.perf_counter()
    t, g = f(x, y, 2)
    print(f"{f.__name__:17s}: ngưỡng = {t:.4f}, gain = {g:.4f} | {(time.perf_counter() - start) * 1000:7.1f} ms")

## 2. Xây cây đệ quy

Tại mỗi node: (1) kiểm tra điều kiện dừng → nếu dừng, tạo **lá**; (2) tìm split tốt nhất trên **mọi feature**; (3) chia dữ liệu, đệ quy cho hai con.

In [ ]:
@dataclass
class Node:
    proba: np.ndarray                    # phân phối lớp tại node (dùng làm dự đoán nếu là lá)
    feature: int | None = None
    threshold: float | None = None
    left: "Node | None" = None
    right: "Node | None" = None


class SimpleTreeClassifier:
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1):
        self.max_depth, self.min_samples_split, self.min_samples_leaf = max_depth, min_samples_split, min_samples_leaf

    def fit(self, X, y):
        self.classes_, y_enc = np.unique(y, return_inverse=True)
        self.n_features_ = X.shape[1]
        self.feature_importances_ = np.zeros(self.n_features_)
        self.root_ = self._grow(np.asarray(X, float), y_enc, depth=0)
        self.feature_importances_ /= self.feature_importances_.sum()
        return self

    def _grow(self, X, y, depth):
        counts = np.bincount(y, minlength=len(self.classes_))
        node = Node(proba=counts / counts.sum())
        if (self.max_depth is not None and depth >= self.max_depth) or len(y) < self.min_samples_split \
                or counts.max() == len(y):                                       # node thuần → dừng
            return node
        best = (None, None, 0.0)
        for j in range(self.n_features_):
            if np.unique(X[:, j]).size < 2:
                continue
            t, gain = best_split_fast(X[:, j], y, len(self.classes_))
            n_left = np.sum(X[:, j] <= t)
            if gain > best[2] and min(n_left, len(y) - n_left) >= self.min_samples_leaf:
                best = (j, t, gain)
        if best[0] is None:
            return node
        j, t, gain = best
        self.feature_importances_[j] += gain * len(y)          # MDI: tổng giảm impurity có trọng số
        mask = X[:, j] <= t
        node.feature, node.threshold = j, t
        node.left = self._grow(X[mask], y[mask], depth + 1)
        node.right = self._grow(X[~mask], y[~mask], depth + 1)
        return node

    def _predict_one(self, x):
        node = self.root_
        while node.feature is not None:
            node = node.left if x[node.feature] <= node.threshold else node.right
        return node.proba

    def predict_proba(self, X):
        return np.array([self._predict_one(x) for x in np.asarray(X, float)])

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

    def to_text(self, names, node=None, indent=""):
        node = node or self.root_
        if node.feature is None:
            return f"{indent}→ {self.classes_[np.argmax(node.proba)]}  (p = {node.proba.round(2)})\n"
        return (f"{indent}if {names[node.feature]} <= {node.threshold:.2f}:\n" + self.to_text(names, node.left, indent + "    ")
                + f"{indent}else:\n" + self.to_text(names, node.right, indent + "    "))

> ⚠️ Phiên bản này dùng Gini và chọn split tốt nhất **trước**, rồi mới kiểm tra `min_samples_leaf`. scikit-learn chỉ xét các split thoả `min_samples_leaf` ngay từ đầu (xem `src/ml_course/algorithms/tree.py`) — đủ gần để học nguyên lý.

## 3. Thử trên dữ liệu iris

In [ ]:
iris = pd.read_csv("../../data/sample/iris.csv")
features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X, y = iris[features].to_numpy(), iris["species"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

ours = SimpleTreeClassifier(max_depth=3).fit(X_train, y_train)
sk = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print(ours.to_text(features))
print(f"accuracy test: scratch = {np.mean(ours.predict(X_test) == y_test):.4f} | sklearn = {sk.score(X_test, y_test):.4f}")
print("feature importance (scratch):", dict(zip(features, ours.feature_importances_.round(3))))
print("feature importance (sklearn):", dict(zip(features, sk.feature_importances_.round(3))))

Cây tìm ra đúng điều ta thấy ở chương 09: `petal_length`/`petal_width` gần như đủ để phân loại; `sepal_*` gần như không được dùng.

> 🔎 Hai cây khác nhau ngay ở gốc: `petal_length <= 2.35` và `petal_width <= 0.8` đều tách **hoàn hảo** setosa → gain bằng nhau (hoà). Code của ta chọn feature đầu tiên đạt max; sklearn duyệt feature theo thứ tự ngẫu nhiên (`random_state`) nên chọn feature kia. Dự đoán gần như giống nhau, nhưng **feature importance khác hẳn** (0.57/0.43 so với 0.03/0.97) — cảnh báo sớm: importance của các feature tương quan mạnh rất không ổn định, đừng diễn giải nó như sự thật tuyệt đối (bài 05).

**Feature importance (MDI — Mean Decrease in Impurity):** tổng mức giảm impurity (nhân số mẫu) tại mọi node dùng feature đó, chuẩn hoá về tổng 1. Nhanh nhưng có thiên lệch (bài 05).

## 4. Độ sâu và decision boundary

In [ ]:
Xm, ym = make_classification_2d("moons", n=400, noise=0.3, seed=1)
fig, axes = plt.subplots(1, 4, figsize=(22, 4.6))
for ax, depth in zip(axes, [1, 3, 6, None]):
    model = SimpleTreeClassifier(max_depth=depth).fit(Xm, ym)
    plot_decision_boundary(model, Xm, ym, ax=ax)
    ax.set_title(f"max_depth = {depth} | train acc = {np.mean(model.predict(Xm) == ym):.2f}")
plt.show()

Không giới hạn độ sâu → cây chia tới khi mọi lá **thuần** → train accuracy = 100%, ranh giới bám theo từng điểm nhiễu (overfit — bài 04).

## 5. Độ phức tạp

| | Chi phí |
|---|---|
| Tìm split tại một node có $n$ mẫu | $O(d \cdot n\log n)$ (sắp xếp mỗi feature) |
| Train cả cây (cân bằng) | ~$O(d \cdot n\log^2 n)$ |
| Dự đoán một mẫu | $O(\text{độ sâu})$ — rất nhanh |

## 6. Phiên bản tham chiếu

In [ ]:
from ml_course.algorithms import DecisionTreeScratch

ref = DecisionTreeScratch(max_depth=3).fit(X_train, y_train)
print(ref.export_text(features))
print(f"accuracy test = {np.mean(ref.predict(X_test) == y_test):.4f}")

## 🧠 Kiểm tra nhanh

1. Vì sao ngưỡng ứng viên chỉ cần là trung điểm giữa hai giá trị liên tiếp (đã sắp xếp)?
2. Kỹ thuật `cumsum` giảm độ phức tạp tìm split từ bao nhiêu xuống bao nhiêu?
3. Liệt kê 4 điều kiện dừng phát triển cây.
4. Dự đoán của cây cho một mẫu tốn bao nhiêu phép so sánh?

## 📌 Tóm tắt

- CART: đệ quy, tham lam; mỗi node thử mọi feature × mọi ngưỡng; split nhị phân $x_j \le t$.
- Sắp xếp + cumsum → tìm split $O(n\log n)$ mỗi feature.
- MDI feature importance = tổng giảm impurity có trọng số.
- Không giới hạn → cây "thuộc lòng" dữ liệu train.